In [ ]:
# Block 1 — setup + read IPUMS XML schema + read raw .dat
import xml.etree.ElementTree as ET
from pyspark.sql import SparkSession, functions as F

#spark = SparkSession.builder.getOrCreate()

spark = SparkSession.builder \
    .config("spark.driver.memory", "2g") \
    .config("spark.executor.memory", "18g") \
    .config("spark.executor.instances", 7) \
    .getOrCreate()

xml_path = "../../evargasnavarro/shared/processed/usa_00001.xml"
dat_path = "../../evargasnavarro/shared/processed/usa_00001.dat"

# Parse IPUMS DDI XML for fixed-width specs
ns = {"ddi": "ddi:codebook:2_5"}
root = ET.parse(xml_path).getroot()

specs = []
for var in root.findall(".//ddi:dataDscr/ddi:var", ns):
    loc = var.find("ddi:location", ns)
    fmt = var.find("ddi:varFormat", ns)
    specs.append({
        "name": var.attrib["name"],
        "start": int(loc.attrib["StartPos"]),
        "width": int(loc.attrib["width"]),
        "dcml": int(var.attrib.get("dcml", "0")),
        "type": fmt.attrib.get("type", "character") if fmt is not None else "character"
    })

raw_df = spark.read.text(dat_path)
raw_df.show(5, truncate=False)  # raw fixed-width lines
print(f"Columns in XML spec: {len(specs)}")

+---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

In [2]:
# Block 2 — parse to Spark DataFrame + cast numerics + inspect
df = raw_df.select(*[
    F.substring("value", s["start"], s["width"]).alias(s["name"])
    for s in specs
])

for s in specs:
    if s["type"] == "numeric":
        df = df.withColumn(s["name"], F.col(s["name"]).cast("double"))
        if s["dcml"] > 0:
            df = df.withColumn(s["name"], F.col(s["name"]) / (10 ** s["dcml"]))

# readable checks
df.select("YEAR", "STATEFIP", "SEX", "AGE", "RACE", "EDUC", "INCTOT").show(10, truncate=False)
df.printSchema()

+------+--------+---+----+----+----+---------+
|YEAR  |STATEFIP|SEX|AGE |RACE|EDUC|INCTOT   |
+------+--------+---+----+----+----+---------+
|2001.0|1.0     |1.0|39.0|1.0 |3.0 |6400.0   |
|2001.0|1.0     |1.0|33.0|2.0 |10.0|30000.0  |
|2001.0|1.0     |2.0|40.0|1.0 |6.0 |16500.0  |
|2001.0|1.0     |1.0|10.0|1.0 |1.0 |9999999.0|
|2001.0|1.0     |2.0|21.0|1.0 |3.0 |0.0      |
|2001.0|1.0     |1.0|39.0|1.0 |6.0 |21000.0  |
|2001.0|1.0     |2.0|55.0|1.0 |10.0|12000.0  |
|2001.0|1.0     |1.0|38.0|1.0 |3.0 |20000.0  |
|2001.0|1.0     |2.0|32.0|1.0 |2.0 |0.0      |
|2001.0|1.0     |1.0|10.0|1.0 |1.0 |9999999.0|
+------+--------+---+----+----+----+---------+
only showing top 10 rows

root
 |-- YEAR: double (nullable = true)
 |-- SAMPLE: double (nullable = true)
 |-- SERIAL: double (nullable = true)
 |-- CBSERIAL: double (nullable = true)
 |-- NUMPREC: double (nullable = true)
 |-- SUBSAMP: double (nullable = true)
 |-- HHWT: double (nullable = true)
 |-- EXPWTH: double (nullable = true)
 |-- HH

In [3]:
row_count = df.count()
print("Number of Rows:", row_count)
column_count = len(df.columns)
print("Number of Columns:", column_count)

Number of Rows: 67125780
Number of Columns: 238


In [4]:
import requests
import pandas as pd

# Get active Spark Context and REST endpoint
sc = spark.sparkContext
url = f"{sc.uiWebUrl}/api/v1/applications/{sc.applicationId}/executors"
print("Executors endpoint:", url)

# Fetch executor data from Spark REST API
response = requests.get(url, timeout=30)
response.raise_for_status()
executors = response.json()

# Format into a readable table (do not overwrite Spark DataFrame variable name)
exec_df = pd.DataFrame(executors)[['id', 'totalCores', 'maxMemory', 'activeTasks', 'isActive']]
exec_df['maxMemory_GB'] = (exec_df['maxMemory'] / (1024**3)).round(2)

non_driver = exec_df[exec_df['id'] != 'driver']
print("Total executor rows:", len(exec_df))
print("Non-driver executor rows:", len(non_driver))

if len(non_driver) == 0:
    print("No non-driver executors reported. This is expected when runtime master is local[*].")

exec_df

,id,totalCores,maxMemory,activeTasks,isActive,maxMemory_GB
0,driver,8,1099746508,0,True,1.02


In [5]:
print(spark.sparkContext.master)

local[*]


In [6]:
print("master:", spark.sparkContext.master)
print("appId:", spark.sparkContext.applicationId)
print("executor.instances:", spark.conf.get("spark.executor.instances", "NA"))
print("executor.memory:", spark.conf.get("spark.executor.memory", "NA"))

master: local[*]
appId: local-1777271547855
executor.instances: 7
executor.memory: 18g


## Submission Notes (if `master` stays `local[*]`)

If runtime diagnostics still show `master = local[*]`, keep these artifacts for grading:

1. Jupyter request settings screenshot (cores + memory)
2. Expanse Active Jobs screenshot (allocated CPUs/memory, RUNNING)
3. Notebook diagnostics output (`master`, `appId`, configured executor values)
4. REST executor table from `/api/v1/applications/{appId}/executors`

Suggested TA/instructor confirmation message:

"We requested Expanse resources (8 cores, 128GB), used formula-based Spark config (`driver=2g`, `executor.instances=7`, `executor.memory=18g`), and verified with REST API. Runtime still reports `master=local[*]` and executors endpoint returns driver-only in this container. Can you confirm this evidence is acceptable for Part 2.d, or share the required non-local startup method for this environment?"